# DS-TGNN V1.1: Deep-Dive Evaluation Report

This notebook provides the visual proof for each of the three architectural upgrades:
1. **Experiment 1 (Embargo)**: Comparison of convergence with/without temporal gaps.
2. **Experiment 2 (MC-Dropout)**: Uncertainty calibration analysis.
3. **Experiment 3 (Weighted Loss)**: Performance breakdown by return magnitude.

---

In [ ]:
# --- 📦 STEP 1: REMOTE BOOTSTRAP (Install & Define) ---
try:
    if 'google.colab' in str(get_ipython()):
        print("📦 Installing torch-geometric on remote server (this takes ~30s)... ")
        !pip install torch_geometric -q
except NameError: pass

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
from torch_geometric.nn import GATv2Conv

# Define Dataset w/ Embargo
class CommodityWeatherDataset(Dataset):
    def __init__(self, steps=2000, N=8, T=180, F=5):
        super().__init__(); torch.manual_seed(42)
        self.tl = torch.randn(steps, N, F)
        self.rt = torch.randn(steps, N) * 0.05
    def __len__(self): return len(self.tl) - 210
    def __getitem__(self, i): return self.tl[i:i+180].transpose(0, 1), self.rt[i+180:i+210].sum(dim=0)

def get_dataloaders(use_embargo=True, batch_size=32):
    ds = CommodityWeatherDataset()
    vl = len(ds); train_end = int(vl * 0.6)
    gap = 45 if use_embargo else 0
    val_start, val_end = train_end + gap, train_end + gap + int(vl * 0.2)
    test_start = val_end + gap
    return (DataLoader(Subset(ds, list(range(train_end))), batch_size=batch_size, shuffle=True),
            DataLoader(Subset(ds, list(range(test_start, vl))), batch_size=batch_size))

# Define Models
class Diffusion(nn.Module):
    def __init__(self, input_dim=3600):
        super().__init__(); self.mlp = nn.Sequential(nn.Linear(input_dim+16, 128), nn.ReLU(), nn.Linear(128, input_dim))
        self.t_mlp = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 16))
    def forward(self, x, t): return self.mlp(torch.cat([x, self.t_mlp(t.view(-1, 1))], dim=-1))

class DiffusionReturnPrediction(nn.Module):
    def __init__(self, denoiser, F=5, lst_h=64, gnn_h=32):
        super().__init__(); self.denoiser = denoiser
        self.lstm = nn.LSTM(F, lst_h, batch_first=True)
        self.gnn = GATv2Conv(lst_h, gnn_h, heads=1)
        self.mlp = nn.Sequential(nn.Linear(gnn_h, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))
    def forward(self, x, edge):
        B, N, T, F_ = x.shape; x_f = x.reshape(B*N, T, F_); _, (h, _) = self.lstm(x_f)
        h = h.squeeze(0).view(B, N, -1); g_out = torch.stack([self.gnn(h[i], edge) for i in range(B)])
        return self.mlp(g_out).squeeze(-1)
    def enable_dropout(self): 
        for m in self.modules(): 
            if isinstance(m, nn.Dropout): m.train()
print("✅ Setup Complete.")

In [ ]:
# --- 🧪 STEP 2: MULTI-RUN EXECUTION ---
def run_suite(use_embargo, use_weighted, mc_samples=1):
    train_loader, test_loader = get_dataloaders(use_embargo=use_embargo)
    edge = torch.tensor([[0,1,2,3,4,0,1,5],[3,4,0,1,2,5,6,7]], dtype=torch.long)
    model = DiffusionReturnPrediction(Diffusion()).train(); opt = torch.optim.Adam(model.parameters(), lr=0.002)
    losses = []
    for _ in range(5): 
        ep_l = []
        for x, y in train_loader:
            opt.zero_grad(); p = model(x, edge); loss = (p-y)**2
            if use_weighted: loss = loss * (1.0 + torch.abs(y)*15.0)
            loss = loss.mean(); loss.backward(); opt.step(); ep_l.append(loss.item())
        losses.append(np.mean(ep_l))
    model.eval()
    if mc_samples > 1: 
        model.enable_dropout()
    pts, stds, ys = [], [], []
    with torch.no_grad():
        for x, y in test_loader:
            bp = torch.stack([model(x, edge) for _ in range(mc_samples)])
            pts.append(bp.mean(dim=0)); stds.append(bp.std(dim=0)); ys.append(y)
    p_c, s_c, y_c = torch.cat(pts).numpy().ravel(), torch.cat(stds).numpy().ravel(), torch.cat(ys).numpy().ravel()
    return {"losses": losses, "preds": p_c, "stds": s_c, "targets": y_c}

print("🚀 Running Baseline (A) vs Embargo (B) vs Full V1.1 (D)... ")
res_a = run_suite(False, False, 1)
res_b = run_suite(True, False, 1)
res_d = run_suite(True, True, 20)
print("🏁 Runs Complete.")

In [ ]:
# --- 📊 STEP 3: DEEP DIVE VISUALIZATIONS ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. LOSS COMPARISON (Exp 1: Embargo Effect)
axes[0].plot(res_a['losses'], label='Baseline (A)')
axes[0].plot(res_b['losses'], label='Embargo (B)', linestyle='--')
axes[0].set_title("Experiment 1: Embargo Convergence Impact"); axes[0].set_ylabel("Loss"); axes[0].legend()

# 2. UNCERTAINTY CALIBRATION (Exp 2: MC-Dropout)
error = np.abs(res_d['preds'] - res_d['targets'])
stds = res_d['stds']
axes[1].scatter(stds, error, alpha=0.3, s=10)
m, b = np.polyfit(stds, error, 1)
corr = np.corrcoef(stds, error)[0,1]
axes[1].plot(stds, m*stds + b, color='red', label=f"Corr: {corr:.4f}")
axes[1].set_title("Experiment 2: Uncertainty-Error Calibration"); axes[1].set_xlabel("Pred Std (Uncertainty)"); axes[1].set_ylabel("Abs Error"); axes[1].legend()

# 3. TAIL RMSE IMPACT (Exp 3: Weighted Loss)
bins = [0.0, 0.02, 0.04, 0.06, 0.08, 0.1]
def get_binwise_rmse(preds, targets, bins):
    results = []
    for b in bins[:-1]:
        mask = (np.abs(targets) >= b) & (np.abs(targets) < b + 0.02)
        if np.sum(mask) > 0:
            results.append(np.sqrt(np.mean((preds[mask] - targets[mask])**2)))
        else:
            results.append(0.0)
    return results
rmse_b = get_binwise_rmse(res_b['preds'], res_b['targets'], bins)
rmse_d = get_binwise_rmse(res_d['preds'], res_d['targets'], bins)
axes[2].bar(np.arange(len(bins)-1)-0.2, rmse_b, width=0.4, label='Standard MSE (B)')
axes[2].bar(np.arange(len(bins)-1)+0.2, rmse_d, width=0.4, label='Weighted Loss (D)', color='orange')
axes[2].set_xticks(np.arange(len(bins)-1)); axes[2].set_xticklabels([f'>{b}' for b in bins[:-1]])
axes[2].set_title("Experiment 3: Performance on Tail Returns"); axes[2].set_ylabel("RMSE"); axes[2].legend()

plt.tight_layout(); plt.show()

# PRINT COMPARATIVE TABLE
def final_row(res, name):
    rmse = np.sqrt(np.mean((res['preds']-res['targets'])**2))
    tail_mask = np.abs(res['targets']) > 0.06
    tail = np.sqrt(np.mean((res['preds'][tail_mask]-res['targets'][tail_mask])**2)) if np.sum(tail_mask) > 0 else 0.0
    return {"Experiment": name, "Overall RMSE": rmse, "Tail RMSE (Abs > 0.06)": tail}
df = pd.DataFrame([final_row(res_a, "Baseline"), final_row(res_b, "Embargo Only"), final_row(res_d, "Full V1.1 Suite")])
print("\n--- 📜 FINAL PERFORMANCE SUMMARY ---")
print(df.to_markdown(index=False))